# 📦 Citadel Publish Contract - Testing Center

## Overview

Publish and validate **three protected AI assets** through the AI Hub Gateway using the **Citadel Publish Contract**, then confirm reachability, policy application, usage tracking, and resiliency.

| # | Asset | `assetType` | Result |
|---|---|---|---|
| 1 | Weather Tool | `mcp-from-api` | Sample `weather-api` exposed as an MCP tool server |
| 2 | Microsoft Learn Tool | `mcp-existing` | Remote MS Learn MCP server published + protected |
| 3 | HR Chat Agent | `a2a` | Foundry-hosted agent published as a native A2A endpoint |

## Expected outcomes

- Three assets published via Bicep (`az deployment sub create`)
- MCP `initialize` + `tools/list` handshakes succeed through the gateway
- The A2A agent card is retrievable at `/.well-known/agent.json` and a JSON-RPC `message/send` routes to Foundry
- `mcp-usage` / `a2a-usage` custom metrics appear in Application Insights
- Each remote MCP backend has a native circuit breaker attached
- The published **HR agent** answers an HR question via gateway-routed A2A, and a direct MCP `tools/call` returns a result

## Azure Prerequisites

- An existing Citadel Governance Hub deployment (APIM + App Insights + Cosmos + usage Logic Apps)
- The sample APIs deployed (`isMCPSampleDeployed = true`) so `weather-api` exists as the API→MCP source
- A Foundry **prompt agent** to expose via A2A (see `citadel-agent-frameworks-tests.ipynb` to create one)
- Permission to assign roles on the Foundry project (Owner / User Access Administrator) — step 3️⃣.1 grants the APIM identity **Foundry access** automatically
- Azure credentials with permission to deploy at subscription scope

> **Phase 1 note:** publishing does not create APIM products/subscriptions (that is Access Contract / phase 2). To call `subscriptionRequired` assets, this notebook provisions a **temporary test product + subscription** purely to obtain an `api-key` — a stand-in for what an Access Contract will do in phase 2.

<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

Set `init_from_azd = True` to autoload the governance hub RG/location/subscription from your active `azd` environment, or `False` to fill the `REPLACE` values manually. Provide the Foundry agent coordinates for the A2A asset.

In [ ]:
import os, sys, json, time, uuid, requests
sys.path.insert(1, '../shared')
import utils
from apimtools import APIMClientTool

# ============================================================================
# 🔧 INITIALIZATION MODE
# ============================================================================
init_from_azd = True   # Set False to fill the REPLACE values below manually.

# ============================================================================
# 🔧 GOVERNANCE HUB CONFIGURATION
# ============================================================================
governance_hub_resource_group = "REPLACE"   # RG of the deployed Citadel Governance Hub
location = "REPLACE"                         # Azure region (e.g. "swedencentral")
subscription_id = "REPLACE"                  # Subscription hosting the hub

# ============================================================================
# 🤖 FOUNDRY AGENT (A2A target) CONFIGURATION
# The A2A asset republishes an EXISTING Foundry prompt agent with incoming A2A enabled.
# ============================================================================
enable_a2a_asset      = True
foundry_account_name  = "aif-citadel-agent-08"   # e.g. aif-citadel-agent-08
foundry_project_name  = "proj-citadel-agent-08"   # e.g. proj-citadel-agent-08
foundry_agent_name    = "HR-ChatAgent"   # e.g. HR-ChatAgent

# ============================================================================
# 🧪 TEST HARNESS
# ============================================================================
publish_deployment_name = "citadel-publish-contracts-validation"
test_product_id = "PUBLISH-CONTRACT-TEST"   # temporary product used only to mint an api-key

def _is_unset(v):
    return v is None or v == "" or v == "REPLACE"

if init_from_azd:
    utils.print_info("Loading configuration from azd environment...")
    loaded = utils.load_azd_env({
        "resource_group":  ["AZURE_RESOURCE_GROUP", "GOVERNANCE_HUB_RESOURCE_GROUP"],
        "location":        ["AZURE_LOCATION", "LOCATION"],
        "subscription_id": ["AZURE_SUBSCRIPTION_ID"],
        "ai_foundry_services": (["AI_FOUNDRY_SERVICES"], "json"),
    }, verbose=False)
    if _is_unset(governance_hub_resource_group) and "resource_group" in loaded:
        governance_hub_resource_group = loaded["resource_group"]
    if _is_unset(location) and "location" in loaded:
        location = loaded["location"]
    if _is_unset(subscription_id) and "subscription_id" in loaded:
        subscription_id = loaded["subscription_id"]
    if "ai_foundry_services" in loaded and isinstance(loaded["ai_foundry_services"], list) and loaded["ai_foundry_services"]:
        first = loaded["ai_foundry_services"][0]
        if _is_unset(foundry_account_name):
            foundry_account_name = first.get("cognitiveServiceName") or first.get("name") or foundry_account_name
        if _is_unset(foundry_project_name):
            ep = first.get("foundryProjectEndpoint", "")
            if "/projects/" in ep:
                foundry_project_name = ep.rstrip("/").rsplit("/projects/", 1)[-1]

utils.print_ok(f"Resource group : {governance_hub_resource_group}")
utils.print_ok(f"Location       : {location}")
utils.print_ok(f"Subscription   : {subscription_id}")
utils.print_ok(f"Foundry agent  : {foundry_account_name}/{foundry_project_name}/{foundry_agent_name} (a2a={enable_a2a_asset})")
utils.print_ok("Notebook variables initialized!")

<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")
if output.success and output.json_data:
    utils.print_info(f"Current user: {output.json_data['user']['name']}")
    utils.print_info(f"Subscription ID: {output.json_data['id']}")
    if _is_unset(subscription_id):
        subscription_id = output.json_data['id']
    if subscription_id != output.json_data['id']:
        utils.print_warning(f"Active subscription differs from configured ({subscription_id}). Run: az account set --subscription {subscription_id}")

<a id='2'></a>
### 2️⃣ Initialize APIM Client Tool

Discover the deployed APIM instance and its gateway URL.

In [ ]:
apimClientTool = APIMClientTool(governance_hub_resource_group)
apimClientTool.initialize()
apim_name = apimClientTool.apim_resource_name
gateway_url = str(apimClientTool.apim_resource_gateway_url)
utils.print_ok(f"APIM: {apim_name}")
utils.print_ok(f"Gateway URL: {gateway_url}")

<a id='3'></a>
### 3️⃣ Enable incoming A2A on the Foundry agent

Foundry prompt agents support the responses protocol, but the **A2A endpoint must be activated** with a `PATCH` that sets the agent card and enables the `a2a` protocol. This mirrors [`local/contracts/a2a-foundry-sample-agent.ps1`](../local/contracts/a2a-foundry-sample-agent.ps1) and the Learn guide: [Enable incoming A2A on a Foundry agent](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/enable-agent-to-agent-endpoint).

> Requires the **Foundry User** role on the project. Skipped automatically when `enable_a2a_asset = False`.

In [ ]:
a2a_card_backend_url = ""
a2a_jsonrpc_backend_url = ""
if enable_a2a_asset and not (_is_unset(foundry_account_name) or _is_unset(foundry_project_name) or _is_unset(foundry_agent_name)):
    base_url = f"https://{foundry_account_name}.services.ai.azure.com/api/projects/{foundry_project_name}"
    a2a_card_backend_url = f"{base_url}/agents/{foundry_agent_name}/endpoint/protocols/a2a/agentCard/v1.0"
    a2a_jsonrpc_backend_url = f"{base_url}/agents/{foundry_agent_name}/endpoint/protocols/a2a"
    token = utils.run("az account get-access-token --resource https://ai.azure.com --query accessToken -o tsv").text.strip()
    body = {
        "agent_card": {
            "description": "HR Chat Agent published via the AI Hub Gateway (A2A).",
            "version": "1.0",
            "skills": [{"id": "general-qa", "name": "General Q&A", "description": "Answers HR policy questions"}]
        },
        "agent_endpoint": {"protocol_configuration": {"responses": {}, "a2a": {}}}
    }
    r = requests.patch(f"{base_url}/agents/{foundry_agent_name}?api-version=v1",
                       headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
                       data=json.dumps(body), timeout=60)
    if r.status_code < 300:
        utils.print_ok(f"Incoming A2A enabled on {foundry_agent_name}")
        utils.print_info(f"Agent card backend: {a2a_card_backend_url}")
    else:
        utils.print_error(f"A2A enable failed ({r.status_code}): {r.text[:400]}")
else:
    utils.print_warning("A2A asset disabled or Foundry agent not fully specified; skipping A2A enablement.")

<a id='3-1'></a>
### 3️⃣.1 Grant the APIM managed identity access on the Foundry project

The published A2A backend authenticates to Foundry with **APIM's user-assigned managed identity** (audience `https://ai.azure.com`) — no keys. For that to work the identity needs a data-plane role on the Foundry project.

This step:
1. Discovers the APIM identity (prefers a **user-assigned** identity; falls back to system-assigned).
2. Grants it the Foundry role on the target project scope.
3. Captures the UAMI **`clientId`**, which is passed to the deployment as `managedIdentityClientId` so the backend embeds MI auth using that specific identity.

> Default role is `Azure AI User`. For least privilege you can switch `foundry_role` to `Foundry Agent Consumer`. Requires Owner / User Access Administrator on the project. Skipped when `enable_a2a_asset = False`.

In [ ]:
# Identity passed to the deployment ('' => APIM system-assigned identity)
apim_mi_client_id = ""
apim_mi_principal_id = ""
foundry_role = "Foundry Agent Consumer"   # least-privilege alternative: "Foundry Agent Consumer"

if enable_a2a_asset and not _is_unset(foundry_account_name):
    # 1) APIM identity — prefer a user-assigned identity, else system-assigned
    idj = utils.run(f"az apim show -g {governance_hub_resource_group} -n {apim_name} --query identity -o json",
                    "Retrieved APIM identity", "Failed to read APIM identity")
    if idj.success and idj.json_data:
        ident = idj.json_data
        uami = ident.get("userAssignedIdentities") or {}
        if uami:
            _rid, _val = next(iter(uami.items()))
            apim_mi_client_id = _val.get("clientId", "")
            apim_mi_principal_id = _val.get("principalId", "")
            utils.print_info(f"APIM user-assigned identity: clientId={apim_mi_client_id}")
        elif ident.get("principalId"):
            apim_mi_principal_id = ident["principalId"]
            utils.print_info("APIM system-assigned identity (managedIdentityClientId left empty)")

    # 2) Foundry project resource id (account may live in a different resource group)
    foundry_project_id = ""
    acc = utils.run(f"az cognitiveservices account list --query \"[?name=='{foundry_account_name}'].id\" -o tsv")
    account_id = acc.text.strip() if acc.success else ""
    if account_id:
        foundry_project_id = f"{account_id}/projects/{foundry_project_name}"

    # 3) Grant the role so APIM's identity can call the Foundry agent
    if apim_mi_principal_id and foundry_project_id:
        ra = utils.run(
            f"az role assignment create --assignee-object-id {apim_mi_principal_id} "
            f"--assignee-principal-type ServicePrincipal --role \"{foundry_role}\" --scope {foundry_project_id}",
            f"Granted '{foundry_role}' to APIM identity on the Foundry project",
            "Role assignment failed (may already exist, or you lack Owner/User Access Administrator)")
        if not ra.success:
            utils.print_warning("If the role name is unavailable in your tenant, try 'Foundry Agent Consumer' or assign it in the portal.")
    else:
        utils.print_warning("Could not resolve APIM principalId or Foundry project id; grant the Foundry role manually.")
else:
    utils.print_info("A2A asset disabled; skipping Foundry access grant.")

<a id='4'></a>
### 4️⃣ Publish the assets (deploy the Publish Contract)

Generate a `.bicepparam` with the three assets and deploy the `citadel-publish-contracts` module at subscription scope.

In [ ]:
assets = [
    {
        "assetType": "mcp-from-api", "name": "weather-tool", "displayName": "Weather Tool (MCP)",
        "description": "Weather data operations, published as an MCP tool server.", "path": "weather-tool-mcp",
        "metadata": {"version": "1.0.0", "owner": "Platform Engineering", "classification": "internal"},
        "sourceApiName": "weather-api", "operationNames": ["get-weather"], "publishToApiCenter": False,
    },
    {
        "assetType": "mcp-existing", "name": "ms-learn-tool", "displayName": "Microsoft Learn Tool (MCP)",
        "description": "Microsoft Learn MCP server published through the gateway.", "path": "ms-learn-tool-mcp",
        "transportType": "streamable", "subscriptionRequired": True,
        "metadata": {"version": "1.0.0", "owner": "Knowledge Mgmt", "classification": "public"},
        "backend": {"url": "https://learn.microsoft.com/api/mcp", "authType": "none"}, "publishToApiCenter": False,
    },
]
if enable_a2a_asset and a2a_card_backend_url:
    assets.append({
        "assetType": "a2a", "name": "hr-chat-agent", "displayName": "HR Chat Agent (A2A)",
        "description": "HR assistant published via A2A.", "path": "hr-chat-agent",
        "agentId": foundry_agent_name, "subscriptionRequired": True, "subscriptionKeyHeaderName": "api-key", "agentCardPath": "/.well-known/agent.json",
        "agentCardBackendUrl": a2a_card_backend_url, "jsonRpcPath": "/",
        "metadata": {"version": "1.0.0", "owner": "HR Digital", "classification": "confidential"},
        "backend": {"url": a2a_jsonrpc_backend_url, "authType": "managed-identity", "authConfig": {"resource": "https://ai.azure.com"}},
        "publishToApiCenter": False,
    })

def _bicep(v, ind=0):
    pad = "  " * ind
    if isinstance(v, bool):
        return "true" if v else "false"
    if isinstance(v, (int, float)):
        return str(v)
    if isinstance(v, str):
        return "'" + v.replace("'", "\\'") + "'"
    if isinstance(v, list):
        return "[\n" + "".join(f"{pad}  {_bicep(i, ind+1)}\n" for i in v) + f"{pad}]"
    if isinstance(v, dict):
        return "{\n" + "".join(f"{pad}  {k}: {_bicep(val, ind+1)}\n" for k, val in v.items()) + f"{pad}}}"
    return "''"

param_text = (
    "using '../../bicep/infra/citadel-publish-contracts/main.bicep'\n\n"
    f"param apim = {{\n  subscriptionId: '{subscription_id}'\n  resourceGroupName: '{governance_hub_resource_group}'\n  name: '{apim_name}'\n}}\n\n"
    # APIM user-assigned identity clientId so the A2A backend authenticates to Foundry as that UAMI
    f"param managedIdentityClientId = '{globals().get('apim_mi_client_id', '')}'\n"
    "param configureCircuitBreaker = true\n\n"
    f"param publishAssets = {_bicep(assets)}\n"
)
os.makedirs("./.publish-validation", exist_ok=True)
param_path = "./.publish-validation/publish-validation-local.bicepparam"
with open(param_path, "w", encoding="utf-8") as f:
    f.write(param_text)
utils.print_info(f"Wrote {param_path} with {len(assets)} assets")
print(param_text)

In [ ]:
cmd = (
    f"az deployment sub create --name {publish_deployment_name} --location {location} "
    f"--template-file ../bicep/infra/citadel-publish-contracts/main.bicep "
    f"--parameters {param_path} -o json"
)
out = utils.run(cmd, "Publish contract deployed", "Publish contract deployment failed", print_output=False)
if out.success and out.json_data:
    published = out.json_data.get("properties", {}).get("outputs", {}).get("publishedAssets", {}).get("value", [])
    for a in published:
        utils.print_ok(f"{a['assetType']:<13} {a['name']:<16} -> {a['endpoint']}")
    globals()['published_assets'] = published

<a id='5'></a>
### 5️⃣ Provision a temporary test product + subscription (api-key)

The published assets are `subscriptionRequired`. Products/subscriptions are Access Contract scope (phase 2), so here we create a **temporary** product that links the three published APIs and a subscription to mint an `api-key` for testing. This is removed in cleanup.

In [ ]:
client = apimClientTool.client
rg, svc = governance_hub_resource_group, apim_name
# All published assets are subscriptionRequired (MCP tools + the A2A agent), so link every API to the product.
api_names = [a["name"] for a in assets]
client.product.create_or_update(rg, svc, test_product_id, {
    "display_name": "Publish Validation", "description": "Temporary product for publish-contract validation",
    "subscription_required": True, "approval_required": False, "state": "published",
})
for name in api_names:
    try:
        client.product_api.create_or_update(rg, svc, test_product_id, name)
        utils.print_info(f"Linked API '{name}' to {test_product_id}")
    except Exception as e:
        utils.print_warning(f"Could not link '{name}': {e}")
sub_id = "publish-validation-sub"
client.subscription.create_or_update(rg, svc, sub_id, {
    "display_name": "Publish Validation Sub", "scope": f"/products/{test_product_id}", "state": "active",
})
secrets = client.subscription.list_secrets(rg, svc, sub_id)
api_key = secrets.primary_key
utils.print_ok(f"Test api-key acquired (product {test_product_id})")

<a id='6'></a>
### 6️⃣ Validate the MCP tools (handshake + tools/list)

Perform the MCP `initialize` handshake and list tools over the streamable HTTP transport for both published tool servers.

> **Endpoint convention:** for an **API→MCP** tool (`mcp-from-api`) APIM appends `/mcp` to the path, so the endpoint is `{gateway}/{path}/mcp`. For a **native/remote MCP** server (`mcp-existing`) APIM does **not** append `/mcp` — the endpoint is `{gateway}/{path}`.

In [ ]:
def mcp_call(endpoint, api_key, method, params=None, mcp_session=None):
    headers = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream", "api-key": api_key}
    if mcp_session:
        headers["Mcp-Session-Id"] = mcp_session
    # APIM's MCP runtime requires a NUMERIC JSON-RPC id; a non-numeric string (e.g. a UUID) is rejected as 'Invalid JSON payload'.
    mcp_call._id = getattr(mcp_call, "_id", 0) + 1
    payload = {"jsonrpc": "2.0", "id": mcp_call._id, "method": method}
    if params is not None:
        payload["params"] = params
    r = requests.post(endpoint, headers=headers, data=json.dumps(payload), timeout=60)
    text, data = r.text, None
    if "text/event-stream" in r.headers.get("content-type", ""):
        for line in text.splitlines():
            if line.startswith("data:"):
                try:
                    data = json.loads(line[5:].strip()); break
                except Exception:
                    pass
    else:
        try:
            data = r.json()
        except Exception:
            pass
    return r, data

def validate_mcp(endpoint, api_key, label):
    utils.print_info(f"--- {label}: {endpoint}")
    r, data = mcp_call(endpoint, api_key, "initialize", {
        "protocolVersion": "2025-06-18", "capabilities": {},
        "clientInfo": {"name": "citadel-validation", "version": "1.0"}})
    session = r.headers.get("Mcp-Session-Id")
    if r.status_code < 300 and data and "result" in data:
        utils.print_ok(f"{label}: initialize OK (session={session})")
        r2, data2 = mcp_call(endpoint, api_key, "tools/list", {}, mcp_session=session)
        tools = (data2 or {}).get("result", {}).get("tools", []) if data2 else []
        utils.print_ok(f"{label}: tools/list -> {[t.get('name') for t in tools]}")
        return True
    utils.print_error(f"{label}: initialize failed ({r.status_code}) {r.text[:300]}")
    return False

results = {}

def mcp_endpoint(asset):
    # APIM appends /mcp only for API->MCP tools; native/remote MCP servers are served at the path as-is.
    return f"{gateway_url}/{asset['path']}/mcp" if asset['assetType'] == 'mcp-from-api' else f"{gateway_url}/{asset['path']}"

for a in assets:
    if a['assetType'] in ('mcp-from-api', 'mcp-existing'):
        results[a['name']] = validate_mcp(mcp_endpoint(a), api_key, a['displayName'])

<a id='7'></a>
### 7️⃣ Validate the A2A agent (agent card + JSON-RPC)

Fetch the agent card that APIM re-exposes at `/.well-known/agent.json`, then send a JSON-RPC `message/send` that the gateway proxies (with its managed identity) to the Foundry agent.

In [ ]:
if enable_a2a_asset and a2a_card_backend_url:
    agent_base = f"{gateway_url}/hr-chat-agent"
    card_url = f"{agent_base}/.well-known/agent.json"
    utils.print_info(f"Validating A2A agent card at {card_url}...")
    # The A2A API is subscriptionRequired; present the test api-key in the configured header.
    rc = requests.get(card_url, headers={"api-key": api_key}, timeout=60)
    if rc.status_code < 300:
        utils.print_ok(f"Agent card reachable (HTTP {rc.status_code}) via {card_url}")
        results['hr-chat-agent-card'] = True
    else:
        utils.print_error(f"Agent card fetch failed ({rc.status_code}): {rc.text[:300]}")
        results['hr-chat-agent-card'] = False

    # Foundry serves A2A v0.3 by default; the message object requires a 'kind' field. Numeric JSON-RPC id.
    rpc = {"jsonrpc": "2.0", "id": 1, "method": "message/send",
           "params": {"message": {"kind": "message", "role": "user", "messageId": str(uuid.uuid4()),
                                   "parts": [{"kind": "text", "text": "What can you help me with?"}]}}}
    rr = requests.post(agent_base, headers={"Content-Type": "application/json", "api-key": api_key},
                       data=json.dumps(rpc), timeout=120)
    if rr.status_code < 300:
        utils.print_ok(f"A2A message/send routed to Foundry (HTTP {rr.status_code})")
        results['hr-chat-agent-rpc'] = True
    else:
        utils.print_warning(f"A2A message/send returned {rr.status_code}: {rr.text[:300]}")
        results['hr-chat-agent-rpc'] = False
else:
    utils.print_warning("A2A asset not published; skipping agent validation.")

<a id='8'></a>
### 8️⃣ Validate usage tracking (custom metrics)

The baseline policies emit `McpRequests` (namespace `mcp-usage`) and `A2ARequests` (namespace `a2a-usage`) on each inbound call. Query Application Insights `customMetrics` to confirm they land (metrics may take a couple of minutes to appear).

In [ ]:
app_insights_name = None
ai_out = utils.run(f"az resource list -g {governance_hub_resource_group} --resource-type Microsoft.Insights/components -o json")
if ai_out.success and ai_out.json_data:
    # Prefer the APIM app insights (name usually contains 'apim')
    comps = ai_out.json_data
    app_insights_name = next((c['name'] for c in comps if 'apim' in c['name'].lower()), comps[0]['name'])
    utils.print_info(f"App Insights: {app_insights_name}")
    kql = "customMetrics | where name in ('McpRequests','A2ARequests') | summarize count=sum(valueSum) by name, tostring(customDimensions['deploymentName']) | order by name asc"
    q = utils.run(f"az monitor app-insights query --app {app_insights_name} -g {governance_hub_resource_group} --analytics-query \"{kql}\" -o json")
    if q.success and q.json_data:
        rows = q.json_data.get('tables', [{}])[0].get('rows', [])
        if rows:
            for row in rows:
                utils.print_ok(f"metric {row[0]} | asset {row[1]} | count {row[2]}")
        else:
            utils.print_warning("No mcp-usage/a2a-usage metrics yet — allow a few minutes after the calls and re-run this cell.")
else:
    utils.print_warning("Could not locate an Application Insights component in the hub RG.")

<a id='9'></a>
### 9️⃣ Validate resiliency (circuit breaker on published backends)

Confirm the remote MCP and A2A assets got a native circuit breaker on their backend.

In [ ]:
for name in [a['name'] for a in assets if a['assetType'] == 'mcp-existing']:
    backend_id = f"{name}-backend"
    uri = f"{apimClientTool.apim_service_id}/backends/{backend_id}?api-version=2024-06-01-preview"
    b = utils.run(f"az rest --method get --uri {uri} -o json")
    if b.success and b.json_data:
        cb = b.json_data.get('properties', {}).get('circuitBreaker')
        if cb and cb.get('rules'):
            rule = cb['rules'][0]
            fc = rule.get('failureCondition', {})
            utils.print_ok(f"{backend_id}: circuit breaker ON (count={fc.get('count')}, interval={fc.get('interval')}, trip={rule.get('tripDuration')})")
        else:
            utils.print_warning(f"{backend_id}: no circuit breaker configured")
    else:
        utils.print_error(f"{backend_id}: backend not found")

<a id='10'></a>
### 🔟 Consume the published assets (Microsoft Agent Framework + direct MCP)

Prove the published assets are usable by a real client, **through the gateway**:
1. A **Microsoft Agent Framework** `A2AAgent` resolves the published **HR agent** card and answers an HR question.
2. A **direct MCP `tools/call`** invokes a published tool through the gateway.

> The publish contract **rewrites the agent card's transport URLs to the gateway**, so the A2A client routes **through the gateway** (presenting the subscription `api-key`) instead of calling Foundry directly. Requires `agent-framework` + `agent-framework-a2a` (version-aligned: `pip install -U agent-framework agent-framework-a2a`).

In [ ]:
# Microsoft Agent Framework: consume the published HR agent (A2A) through the gateway.
# The published agent card advertises the gateway's transport URLs (rewritten by the publish contract),
# so A2AAgent routes through the gateway (presenting the subscription api-key) rather than calling Foundry directly.
import nest_asyncio, asyncio, httpx
from a2a.client import A2ACardResolver
from agent_framework.a2a import A2AAgent
nest_asyncio.apply()

if enable_a2a_asset and a2a_card_backend_url:
    agent_url = f"{gateway_url}/hr-chat-agent"

    async def ask_hr_agent(question):
        # A2A API is subscriptionRequired; present the test api-key on every gateway call (card + JSON-RPC).
        async with httpx.AsyncClient(timeout=120.0, headers={"api-key": api_key}) as http_client:
            resolver = A2ACardResolver(httpx_client=http_client, base_url=agent_url)
            card = await resolver.get_agent_card(relative_card_path="/.well-known/agent.json")
            agent = A2AAgent(name=card.name, description=card.description, agent_card=card, http_client=http_client)
            resp = await agent.run(question)
            return "\n".join(getattr(m, "text", "") for m in resp.messages)

    try:
        utils.print_info("Asking the HR agent (Microsoft Agent Framework -> A2A -> gateway)...")
        answer = asyncio.run(ask_hr_agent("What is our leave policy?"))
        utils.print_ok("HR agent answered:")
        print(answer)
        results['hr-chat-agent-maf'] = bool(answer.strip())
    except Exception as e:
        utils.print_error(f"Agent Framework A2A invocation failed: {e}")
        utils.print_warning("Ensure 'agent-framework' + 'agent-framework-a2a' are installed and version-aligned "
                            "(pip install -U agent-framework agent-framework-a2a).")
        results['hr-chat-agent-maf'] = False
else:
    utils.print_warning("A2A asset not published; skipping Agent Framework A2A consumption.")

In [ ]:
# Direct invocation of a published MCP tool (initialize -> tools/call) through the gateway.
def call_mcp_tool(asset, tool_name, arguments):
    ep = mcp_endpoint(asset)
    r, _ = mcp_call(ep, api_key, "initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                    "clientInfo": {"name": "citadel-validation", "version": "1.0"}})
    session = r.headers.get("Mcp-Session-Id")
    _, data = mcp_call(ep, api_key, "tools/call", {"name": tool_name, "arguments": arguments}, mcp_session=session)
    return data

weather_asset = next((a for a in assets if a["name"] == "weather-tool"), None)
if weather_asset:
    out = call_mcp_tool(weather_asset, "get-weather", {"city": "London"})
    content = (out or {}).get("result", {}).get("content", [])
    text = content[0].get("text", "") if content else json.dumps(out)
    utils.print_ok("Direct MCP tools/call  weather-tool.get-weather('London') ->")
    print(text)
    results['weather-tool-toolcall'] = bool(content)
else:
    utils.print_warning("weather-tool not in assets; skipping direct MCP tool invocation.")

<a id='results'></a>
### 📊 Results Summary

In [ ]:
utils.print_info("=== Publish Contract Validation Summary ===")
for k, v in results.items():
    (utils.print_ok if v else utils.print_error)(f"{k}: {'PASS' if v else 'FAIL'}")
if all(results.values()):
    utils.print_ok("All published assets validated successfully.")
else:
    utils.print_warning("Some checks did not pass — review the sections above.")

<a id='cleanup'></a>
### 🧹 Cleanup (Optional)

Remove the temporary test product/subscription. Set `delete_published_assets = True` to also remove the published APIs/backends.

In [ ]:
delete_published_assets = False
try:
    client.subscription.delete(rg, svc, sub_id, if_match="*")
    client.product.delete(rg, svc, test_product_id, if_match="*", delete_subscriptions=True)
    utils.print_ok("Removed temporary test product + subscription")
except Exception as e:
    utils.print_warning(f"Cleanup of test product failed: {e}")

if delete_published_assets:
    for a in assets:
        try:
            client.api.delete(rg, svc, a['name'], if_match="*")
            utils.print_info(f"Deleted API {a['name']}")
        except Exception as e:
            utils.print_warning(f"Could not delete API {a['name']}: {e}")
        if a['assetType'] in ('mcp-existing', 'a2a'):
            try:
                client.backend.delete(rg, svc, f"{a['name']}-backend", if_match="*")
            except Exception:
                pass
    utils.print_ok("Published assets removed")